# Extract Plant Plot Traits From UAV image

In [1]:
#Require: dsm, orthomosaic image, plot polygon
#ecpect outputs: canopy height(m), canopy area(m2) for every plots
#Jan6, 2026
#yanben shen, yanben@iastate.edu

In [1]:
import math
import glob
import csv
import fileinput
import string
from os.path import join as opj
import arcpy
from arcpy.sa import *
import matplotlib.pyplot as plt
from matplotlib.image import *
import numpy
import pandas as pd
from osgeo import gdal


In [2]:
import sys
print(sys.executable)


C:\Users\PlantBreeding\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\python.exe


# plug in library

# editing the path

In [3]:
import os
img_folder=input('folder of processing')#F:\2025\BrunerYT&earlyCAD\process_folder
os.chdir(img_folder)
arcpy.CheckInExtension('Spatial')
main_folder=img_folder

folder of processing F:\2025\BrunerYT&earlyCAD\process_folder


In [4]:

arcpy.env.overwriteOutput = True

# Pre-requirments


In [5]:
#you need prepare RGB orthomosaic image, DSM from stitching image, and draw the plot polygons for your plot labels
# for image stitching, you can use Pix4D, OpenDroneMap
# for plot polygon, you can use fishnet in ArcGisPro

In [6]:
import os
import arcpy

arcpy.env.overwriteOutput = True

id_field = "plot"   # 🔴 CHANGE if needed

# Loop over each subfolder inside the Process folder
for folder_name in os.listdir(main_folder):
    folder_path = os.path.join(main_folder, folder_name)

    if not os.path.isdir(folder_path):
        continue

    print(f"\nProcessing folder: {folder_name}")

    # Define required file paths
    ortho_path = os.path.join(folder_path, "odm_orthophoto.tif")
    dsm_image  = os.path.join(folder_path, "dsm.tif")
    shp_path   = os.path.join(folder_path, "block_1.shp")
    output_path = os.path.join(folder_path, "output")

    if not os.path.exists(ortho_path) or not os.path.exists(shp_path):
        print(f"  -> Missing ortho or shapefile, skipping.")
        continue

    os.makedirs(output_path, exist_ok=True)

    # ===================== DSM CLIP =====================
    if os.path.exists(dsm_image):
        out_folder_dsm = os.path.join(output_path, "split_dsm_raster")
        os.makedirs(out_folder_dsm, exist_ok=True)

        print("  -> Processing DSM polygons")

        with arcpy.da.SearchCursor(shp_path, ["OID@", id_field, "SHAPE@"]) as cursor:
            for oid, pid, geom in cursor:

                out_name = f"{folder_name}_{pid}.tif"
                out_raster = os.path.join(out_folder_dsm, out_name)

                # ✅ Skip only existing tiles
                if os.path.exists(out_raster):
                    continue

                print(f"     DSM → plot {pid}")

                arcpy.management.Clip(
                    in_raster=dsm_image,
                    rectangle="#",
                    out_raster=out_raster,
                    in_template_dataset=geom,
                    nodata_value=-9999,
                    clipping_geometry="ClippingGeometry",
                    maintain_clipping_extent="NO_MAINTAIN_EXTENT"
                )
    else:
        print("  -> DSM not found; skipping DSM.")

    # ===================== RGB CLIP =====================
    out_folder_rgb = os.path.join(output_path, "crop_img_new")
    os.makedirs(out_folder_rgb, exist_ok=True)

    print("  -> Processing ORTHO polygons")

    with arcpy.da.SearchCursor(shp_path, ["OID@", id_field, "SHAPE@"]) as cursor:
        for oid, pid, geom in cursor:

            out_name = f"{folder_name}_{pid}.tif"
            out_raster = os.path.join(out_folder_rgb, out_name)

            # ✅ Skip only existing tiles
            if os.path.exists(out_raster):
                continue

            print(f"     ORTHO → plot {pid}")

            arcpy.management.Clip(
                in_raster=ortho_path,
                rectangle="#",
                out_raster=out_raster,
                in_template_dataset=geom,
                nodata_value=0,
                clipping_geometry="ClippingGeometry",
                maintain_clipping_extent="NO_MAINTAIN_EXTENT"
            )



Processing folder: Jun12
  -> Processing DSM polygons
     DSM → plot 1
     DSM → plot 2
     DSM → plot 3
     DSM → plot 4
     DSM → plot 5
     DSM → plot 6
     DSM → plot 7
     DSM → plot 8
     DSM → plot 9
     DSM → plot 10
     DSM → plot 11
     DSM → plot 12
     DSM → plot 13
     DSM → plot 14
     DSM → plot 15
     DSM → plot 16
     DSM → plot 17
     DSM → plot 18
     DSM → plot 19
     DSM → plot 20
     DSM → plot 21
     DSM → plot 22
     DSM → plot 23
     DSM → plot 24
     DSM → plot 25
     DSM → plot 26
     DSM → plot 27
     DSM → plot 28
     DSM → plot 29
     DSM → plot 30
     DSM → plot 31
     DSM → plot 32
     DSM → plot 33
     DSM → plot 34
  -> Processing ORTHO polygons
     ORTHO → plot 1
     ORTHO → plot 2
     ORTHO → plot 3
     ORTHO → plot 4
     ORTHO → plot 5
     ORTHO → plot 6
     ORTHO → plot 7
     ORTHO → plot 8
     ORTHO → plot 9
     ORTHO → plot 10
     ORTHO → plot 11
     ORTHO → plot 12
     ORTHO → plot 13
     ORTHO → 

In [9]:
# import os
# import arcpy

# arcpy.env.overwriteOutput = True  # overwrite the problematic raster

# # Paths
# folder_name = "Jun19"
# folder_path = r"F:\2025\BrunerYT&earlyCAD\process_folder\Jun19"
# dsm_image = os.path.join(folder_path, "dsm.tif")
# shp_path = os.path.join(folder_path, "block_1.shp")
# out_folder_dsm = os.path.join(folder_path, "output", "split_dsm_raster")

# # Plot to fix
# plot_id_to_fix = 13
# out_raster = os.path.join(out_folder_dsm, f"{folder_name}_{plot_id_to_fix}.tif")

# # Use the shapefile cursor to get the geometry for plot 13
# id_field = "plot"  # 🔴 change if your shapefile uses a different field

# with arcpy.da.SearchCursor(shp_path, ["OID@", id_field, "SHAPE@"]) as cursor:
#     for oid, pid, geom in cursor:
#         if pid == plot_id_to_fix:
#             print(f"Fixing DSM for plot {pid}...")
#             arcpy.management.Clip(
#                 in_raster=dsm_image,
#                 rectangle="#",
#                 out_raster=out_raster,
#                 in_template_dataset=geom,
#                 nodata_value=-9999,
#                 clipping_geometry="ClippingGeometry",
#                 maintain_clipping_extent="NO_MAINTAIN_EXTENT"
#             )
#             print(f"  -> Plot {pid} fixed: {out_raster}")
#             break
#     else:
#         print(f"Plot {plot_id_to_fix} not found in shapefile.")


Fixing DSM for plot 13...
  -> Plot 13 fixed: F:\2025\BrunerYT&earlyCAD\process_folder\Jun19\output\split_dsm_raster\Jun19_13.tif


In [7]:
for folder_name in os.listdir(main_folder):
    print(folder_name)

Jun12
Jun19
Jun30
Jun6
May13
May29


In [8]:
### inside the output folder, you should see the two folders. crop_img_new has plot-wise RGB and split_dsm_raster has plot-wise DSM   

In [9]:
import os
import csv
import arcpy
from arcpy.sa import *
import cv2
import numpy as np
from osgeo import gdal

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

In [10]:
# ------------------------------------------------------------
# UTIL: read GeoTIFF as (H,W,C) numpy array + geo/proj
# ------------------------------------------------------------
def read_img(filename):
    dataset = gdal.Open(filename)
    if dataset is None:
        raise RuntimeError(f"GDAL could not open: {filename}")

    cols = dataset.RasterXSize
    rows = dataset.RasterYSize
    bands = dataset.RasterCount
    im_proj = dataset.GetProjection()
    im_geotrans = dataset.GetGeoTransform()

    bandlist = []
    for i in range(bands):
        band = dataset.GetRasterBand(i + 1)
        data = band.ReadAsArray(0, 0, cols, rows)
        bandlist.append(data)

    im_data = np.stack(bandlist, axis=2)  # (rows, cols, bands)
    dataset = None
    return im_proj, im_geotrans, im_data


def _safe_div(n, d, eps=1e-12):
    return n / (d + eps)


def _masked_stats(arr, mask, p_low=10, p_high=90):
    vals = arr[mask].astype(np.float64, copy=False)
    vals = vals[np.isfinite(vals)]          # <- ignore NaN/Inf
    if vals.size == 0:
        return {"mean": np.nan, "median": np.nan, "p10": np.nan, "p90": np.nan}

    return {
        "mean": float(np.mean(vals)),
        "median": float(np.median(vals)),
        "p10": float(np.percentile(vals, p_low)),
        "p90": float(np.percentile(vals, p_high)),
    }


In [11]:
# ------------------------------------------------------------
# CLASS: Canopy/SOIL polygons + canopy mask + VI stats
# ------------------------------------------------------------
class Canopy_Soil_Polygon:
    def __init__(self, image_data, geotransform, projection, workspace):
        """
        image_data expected band order: [R,G,B] in channels 0,1,2.
        If your TIFF is [B,G,R], swap before passing in or swap here.
        """
        self.cropped_image_data = image_data
        self.geotransform = geotransform
        self.projection = projection
        arcpy.env.workspace = workspace

        # float bands in 0..1
        self.red_band   = self.cropped_image_data[:, :, 0].astype(np.float32) / 255.0
        self.green_band = self.cropped_image_data[:, :, 1].astype(np.float32) / 255.0
        self.blue_band  = self.cropped_image_data[:, :, 2].astype(np.float32) / 255.0

        # 8-bit bands for CIVE
        self.red_8   = self.cropped_image_data[:, :, 0].astype(np.float32)
        self.green_8 = self.cropped_image_data[:, :, 1].astype(np.float32)
        self.blue_8  = self.cropped_image_data[:, :, 2].astype(np.float32)

        self.exg_img = self.exg()

        # canopy image + boolean mask (True = canopy)
        self.canopy_img, self.canopy_mask = self.process_img()

    def exg(self):
        # Excess Green: 2G - R - B (0..1 scale bands)
        return (2.0 * self.green_band) - self.red_band - self.blue_band

    def process_img(self):
        """
        Build canopy mask:
          - ExG threshold
          - morphology close
          - binary mask
        Returns:
          canopy_img (uint8 RGB with non-canopy set to 0)
          canopy_mask (bool)
        """
        exg_mask = (self.exg_img >= 0.1)  # vegetation threshold (tune as needed)

        img_copy = self.cropped_image_data.copy()
        img_copy[~exg_mask] = 0

        kernel = np.ones((15, 15), np.uint8)
        green = cv2.morphologyEx(img_copy[:, :, 1], cv2.MORPH_CLOSE, kernel)
        _, mask_u8 = cv2.threshold(green, 1, 255, cv2.THRESH_BINARY)

        canopy = cv2.bitwise_and(
            self.cropped_image_data.astype(np.uint8),
            self.cropped_image_data.astype(np.uint8),
            mask=mask_u8.astype(np.uint8)
        )

        canopy_mask = (mask_u8 > 0)
        return canopy, canopy_mask

    def pixel_to_geo(self, x_pix, y_pix):
        x0, px_w, rot1, y0, rot2, px_h = self.geotransform
        X = x0 + x_pix * px_w + y_pix * rot1
        Y = y0 + x_pix * rot2 + y_pix * px_h
        return (X, Y)

    def create_shapefiles_arcpy(self, canopy_shapefile, soil_shapefile):
        # Create spatial reference from projection WKT
        sr = arcpy.SpatialReference()
        sr.loadFromString(self.projection)

        # delete if exists (overwriteOutput doesn't always remove shapefiles cleanly)
        for shp in [canopy_shapefile, soil_shapefile]:
            if arcpy.Exists(shp):
                arcpy.management.Delete(shp)

        # build canopy mask → contours
        mask_u8 = (self.canopy_mask.astype(np.uint8) * 255)
        contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # create outputs
        arcpy.CreateFeatureclass_management(
            os.path.dirname(canopy_shapefile),
            os.path.basename(canopy_shapefile),
            "POLYGON",
            spatial_reference=sr
        )
        arcpy.management.AddField(canopy_shapefile, "id", "LONG")

        arcpy.CreateFeatureclass_management(
            os.path.dirname(soil_shapefile),
            os.path.basename(soil_shapefile),
            "POLYGON",
            spatial_reference=sr
        )
        arcpy.management.AddField(soil_shapefile, "id", "LONG")

        # insert canopy polygons
        with arcpy.da.InsertCursor(canopy_shapefile, ["SHAPE@", "id"]) as cur:
            for cnt in contours:
                cnt = cnt.squeeze()
                if cnt.ndim != 2 or len(cnt) < 3:
                    continue

                ring = arcpy.Array([
                    arcpy.Point(*self.pixel_to_geo(float(pt[0]), float(pt[1])))
                    for pt in cnt
                ])

                poly = arcpy.Polygon(ring, sr)

                # ArcPy has no poly.isEmpty — use area check instead
                if poly is not None and poly.area > 0:
                    cur.insertRow([poly, 1])


        # soil = full extent minus canopy union
        h, w = mask_u8.shape
        full = arcpy.Polygon(arcpy.Array([
            arcpy.Point(*self.pixel_to_geo(0, 0)),
            arcpy.Point(*self.pixel_to_geo(0, h)),
            arcpy.Point(*self.pixel_to_geo(w, h)),
            arcpy.Point(*self.pixel_to_geo(w, 0))
        ]), sr)

        canopy_union = None
        with arcpy.da.SearchCursor(canopy_shapefile, ["SHAPE@"]) as sc:
            for (geom,) in sc:
                if geom is None:
                    continue
                canopy_union = geom if canopy_union is None else canopy_union.union(geom)

        soil_geom = full if canopy_union is None else full.difference(canopy_union)

        with arcpy.da.InsertCursor(soil_shapefile, ["SHAPE@", "id"]) as cur2:
            if soil_geom is not None and soil_geom.area > 0:
                cur2.insertRow([soil_geom, 1])

        print(f"Shapefiles {canopy_shapefile} and {soil_shapefile} created successfully.")
    def compute_canopy_vi_stats(self):
        """
        Compute canopy-only vegetation index stats:
          mean, median, 10th and 90th percentiles
        Returns a dict of flat columns for CSV.
        """
        R = self.red_band
        G = self.green_band
        B = self.blue_band

        R8 = self.red_8
        G8 = self.green_8
        B8 = self.blue_8

        m = self.canopy_mask

        # --- indices (2D arrays) ---
        ExG   = (2.0 * G) - R - B
        ExB   = (1.4 * B) - G
        ExR   = (1.4 * R) - G
        RGR   = _safe_div(R, G)
        BGR   = _safe_div(B, G)
        NGRDI = _safe_div((G - R), (G + R))
        NGBDI = _safe_div((G - B), (G + B))
        MGRVI = _safe_div((G**2 - R**2), (G**2 + R**2))
        RGBVI = _safe_div((G**2 - (R * B)), (G**2 + (R * B)))
        GLI   = _safe_div((2.0 * G - R - B), (2.0 * G + R + B))
        G_only = G.copy()

        # CIVE uses 0..255 bands
        CIVE  = (0.441 * R8) - (0.881 * G8) + (0.385 * B8) + 18.78745

        ExGR  = ExG - ExR

        # requested extras
        GBVI  = _safe_div((G - B), (G + B))          # same as NGBDI
        GRBI  = _safe_div((G - R), (G + R))          # same as NGRDI
        VARI  = _safe_div((G - R), (G + R - B))
        GRVI  = _safe_div((G - R), (G + R))          # same as NGRDI/GRBI
        RBVI  = _safe_div((R - B), (R + B))

        indices = {
            "ExG": ExG,
            "ExB": ExB,
            "ExR": ExR,
            "RGR": RGR,
            "BGR": BGR,
            "NGRDI": NGRDI,
            "NGBDI": NGBDI,
            "MGRVI": MGRVI,
            "RGBVI": RGBVI,
            "GLI": GLI,
            "G": G_only,
            "CIVE": CIVE,
            "ExGR": ExGR,
            "GBVI": GBVI,
            "GRBI": GRBI,
            "VARI": VARI,
            "GRVI": GRVI,
            "RBVI": RBVI,
        }

        out = {}
        out["canopy_pixel_count"] = int(np.count_nonzero(m))
        out["canopy_pixel_fraction"] = float(np.mean(m)) if m.size else np.nan

        for name, arr in indices.items():
            st = _masked_stats(arr, m, p_low=10, p_high=90)
            out[f"{name}_mean"] = st["mean"]
            out[f"{name}_median"] = st["median"]
            out[f"{name}_p10"] = st["p10"]
            out[f"{name}_p90"] = st["p90"]

        return out



In [12]:
# ------------------------------------------------------------
# CLASS: Zonal Statistics → canopy plant height (your logic)
# ------------------------------------------------------------
class CanopyPlantModel:
    def __init__(self, workspace, soil_polygon, canopy_polygon, dsm_image):
        self.workspace = workspace
        self.soil_polygon = soil_polygon
        self.canopy_polygon = canopy_polygon
        self.dsm_image = dsm_image
        arcpy.env.workspace = self.workspace
        self.soil_stats_table = os.path.join(self.workspace, 'soil_stats_table')
        self.canopy_stats_table = os.path.join(self.workspace, 'canopy_stats_table')

    def calculate_canopy_plant_height(self):
        from arcpy.sa import ExtractByMask

        print("Processing Zonal Statistics (P90 canopy - P10 soil):")
        print(f"  Soil Polygon:   {self.soil_polygon}")
        print(f"  Canopy Polygon: {self.canopy_polygon}")
        print(f"  DSM Image:      {self.dsm_image}")

        def _percentile_from_mask(raster_path, polygon_path, p):
            """Extract raster values under polygon and return percentile p (ignores NoData)."""
            r = ExtractByMask(raster_path, polygon_path)

            # Some ArcPy versions support nodata_to_value; handle both safely
            try:
                arr = arcpy.RasterToNumPyArray(r, nodata_to_value=np.nan).astype(np.float64)
            except TypeError:
                arr = arcpy.RasterToNumPyArray(r).astype(np.float64)

            vals = arr[np.isfinite(arr)]
            if vals.size == 0:
                return None
            return float(np.percentile(vals, p))

        def _sum_polygon_area_m2(shp_path):
            """Sum areas of all polygons in shapefile (map units^2; usually meters^2 for UTM)."""
            total = 0.0
            with arcpy.da.SearchCursor(shp_path, ["SHAPE@AREA"]) as cur:
                for (a,) in cur:
                    if a is not None:
                        total += float(a)
            return total

        # --- Compute robust baseline & canopy height ---
        soil_p10 = _percentile_from_mask(self.dsm_image, self.soil_polygon, 10)
        canopy_p90 = _percentile_from_mask(self.dsm_image, self.canopy_polygon, 90)

        canopy_area_m2 = _sum_polygon_area_m2(self.canopy_polygon)

        canopy_data = []
        if soil_p10 is None or canopy_p90 is None:
            print("Could not calculate height: missing DSM values under soil or canopy mask.")
            canopy_data.append({
                "canopy_area_m2": canopy_area_m2,
                "canopy_plant_height_m": "",
                "soil_dsm_p10": soil_p10 if soil_p10 is not None else "",
                "canopy_dsm_p90": canopy_p90 if canopy_p90 is not None else ""
            })
            return canopy_data

        height = canopy_p90 - soil_p10

        canopy_data.append({
            "canopy_area_m2": canopy_area_m2,
            "canopy_plant_height_m": height,
            "soil_dsm_p10": soil_p10,
            "canopy_dsm_p90": canopy_p90
        })

        print(f"  Canopy area (m²): {canopy_area_m2}")
        print(f"  Soil DSM P10:      {soil_p10}")
        print(f"  Canopy DSM P90:    {canopy_p90}")
        print(f"  Height (P90-P10):  {height} m")

        return canopy_data

In [13]:
# ------------------------------------------------------------
# Match DSM tile for a given RGB tile
# ------------------------------------------------------------
def _match_dsm_tile_for_rgb(rgb_tile_path, dsm_tiles_dir):
    rgb_name = os.path.basename(rgb_tile_path)
    candidate = os.path.join(dsm_tiles_dir, rgb_name)
    if os.path.exists(candidate):
        return candidate

    stem = os.path.splitext(rgb_name)[0]
    if "_" in stem:
        suffix = stem.split("_")[-1]
        for fn in os.listdir(dsm_tiles_dir):
            if not fn.lower().endswith(".tif"):
                continue
            s2 = os.path.splitext(fn)[0]
            if s2.endswith(f"_{suffix}"):
                cand2 = os.path.join(dsm_tiles_dir, fn)
                if os.path.exists(cand2):
                    return cand2
    return None


In [14]:
# ------------------------------------------------------------
# Process one date folder plotwise
# ------------------------------------------------------------
def process_one_date_folder_plotwise(date_folder_path, base_process_dir=None):
    out_dir = os.path.join(date_folder_path, "output")
    rgb_dir = os.path.join(out_dir, "crop_img_new")
    dsm_dir = os.path.join(out_dir, "split_dsm_raster")   # make sure your folder name matches

    if not (os.path.isdir(rgb_dir) and os.path.isdir(dsm_dir)):
        print(f"  -> Skipping {date_folder_path}: missing plot tiles in output/.")
        return []

    rows = []
    date_name = os.path.basename(date_folder_path)

    for fn in os.listdir(rgb_dir):
        if not fn.lower().endswith(".tif"):
            continue

        rgb_tile = os.path.join(rgb_dir, fn)
        dsm_tile = _match_dsm_tile_for_rgb(rgb_tile, dsm_dir)

        plot_stem = os.path.splitext(fn)[0]
        plot_out_dir = os.path.join(out_dir, f"plot_{plot_stem}")
        os.makedirs(plot_out_dir, exist_ok=True)

        # Read RGB tile
        try:
            im_proj, im_geotrans, im_data = read_img(rgb_tile)
        except Exception as e:
            print(f"    [RGB read fail] {rgb_tile}: {e}")
            continue

        canopy_shp = os.path.join(plot_out_dir, f"canopy_{plot_stem}.shp")
        soil_shp   = os.path.join(plot_out_dir, f"soil_{plot_stem}.shp")

        # polygons + VI stats
        try:
            csp = Canopy_Soil_Polygon(im_data, im_geotrans, im_proj, plot_out_dir)
            csp.create_shapefiles_arcpy(canopy_shp, soil_shp)
            vi_stats = csp.compute_canopy_vi_stats()
        except Exception as e:
            print(f"    [Polygon/VI fail] {rgb_tile}: {e}")
            continue

        # zonal stats vs DSM
        cph_val = ""
        area_val = ""
        if dsm_tile and os.path.exists(dsm_tile):
            try:
                model = CanopyPlantModel(plot_out_dir, soil_shp, canopy_shp, dsm_tile)
                res = model.calculate_canopy_plant_height()
                if res and len(res) > 0:
                    area_val = res[0].get("canopy_area_m2", "")
                    cph_val  = res[0].get("canopy_plant_height_m", "")
            except Exception as e:
                print(f"    [Zonal fail] DSM:{dsm_tile} | RGB:{rgb_tile}: {e}")
        else:
            print(f"    [No DSM tile] Matching DSM not found for {rgb_tile}")

        # Build one output row
        row = {
            "folder": date_name,
            "plot": plot_stem,
            "canopy_area_m2": area_val,
            "canopy_plant_height_m": cph_val,
            "rgb_tile": rgb_tile,
            "dsm_tile": dsm_tile if dsm_tile else ""
        }

        # Add vegetation indices (mean/median/p10/p90 etc.)
        row.update(vi_stats)

        rows.append(row)

    # Write per-date CSV inside the date folder
    per_date_csv = os.path.join(out_dir, f"{date_name}_plotwise_canopy_height_and_VI.csv")
    _write_rows_csv(per_date_csv, rows)
    print(f"  -> Saved per-date CSV: {per_date_csv}")

    # ALSO write a "master per date" copy in process/date_masters/
    if base_process_dir is not None:
        date_master_dir = os.path.join(base_process_dir, "date_masters")
        os.makedirs(date_master_dir, exist_ok=True)
        per_date_copy = os.path.join(date_master_dir, f"{date_name}_plotwise_canopy_height_and_VI.csv")
        _write_rows_csv(per_date_copy, rows)
        print(f"  -> Saved date-master copy: {per_date_copy}")

    return rows


def _write_rows_csv(csv_path, rows):
    import csv
    import numpy as np

    if not rows:
        # still write an empty file with minimal headers
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["folder", "plot", "canopy_area_m2", "canopy_plant_height_m", "rgb_tile", "dsm_tile"])
        return

    # Stable header order: base first, then canopy pixel info, then indices in a fixed order
    base_cols = ["folder", "plot", "canopy_area_m2", "canopy_plant_height_m", "rgb_tile", "dsm_tile"]
    extra_first = ["canopy_pixel_count", "canopy_pixel_fraction"]

    idx_order = [
        "ExG","ExB","ExR","RGR","BGR","NGRDI","NGBDI","MGRVI","RGBVI","GLI","G","CIVE","ExGR",
        "GBVI","GRBI","VARI","GRVI","RBVI"
    ]
    stat_order = ["mean", "median", "p10", "p90"]
    vi_cols = [f"{idx}_{st}" for idx in idx_order for st in stat_order]

    # include any other keys that appeared (future-proof)
    all_keys = set()
    for r in rows:
        all_keys.update(r.keys())

    known = set(base_cols + extra_first + vi_cols)
    leftovers = sorted([k for k in all_keys if k not in known])

    header = base_cols + extra_first + vi_cols + leftovers

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=header)
        w.writeheader()
        for r in rows:
            w.writerow(r)   # ✅ FIXED (removed the trailing ".")


In [15]:
#below need to edit pathway

In [16]:
# ------------------------------------------------------------
# MAIN: loop all date folders in “process”
# ------------------------------------------------------------
def main():
    base_process_dir = main_folder ## need your edit for the pathway

    master_rows = []
    for name in os.listdir(base_process_dir):
        fpath = os.path.join(base_process_dir, name)
        if not os.path.isdir(fpath):
            continue

        # Skip the aggregation folder if it exists
        if name.lower() in ["date_masters"]:
            continue

        print(f"\n=== Processing date folder (plot-wise): {name} ===")
        rows = process_one_date_folder_plotwise(fpath, base_process_dir=base_process_dir)
        master_rows.extend(rows)

    master_csv = os.path.join(base_process_dir, "master_plotwise_canopy_height_and_VI_results.csv")
    _write_rows_csv(master_csv, master_rows)
    print(f"\n=== Finished. Master CSV: {master_csv} ===")


if __name__ == "__main__":
    main()



=== Processing date folder (plot-wise): Jun12 ===


C:\Users\PlantBreeding\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\osgeo\gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Shapefiles F:\2025\BrunerYT&earlyCAD\process_folder\Jun12\output\plot_Jun12_1\canopy_Jun12_1.shp and F:\2025\BrunerYT&earlyCAD\process_folder\Jun12\output\plot_Jun12_1\soil_Jun12_1.shp created successfully.
Processing Zonal Statistics (P90 canopy - P10 soil):
  Soil Polygon:   F:\2025\BrunerYT&earlyCAD\process_folder\Jun12\output\plot_Jun12_1\soil_Jun12_1.shp
  Canopy Polygon: F:\2025\BrunerYT&earlyCAD\process_folder\Jun12\output\plot_Jun12_1\canopy_Jun12_1.shp
  DSM Image:      F:\2025\BrunerYT&earlyCAD\process_folder\Jun12\output\split_dsm_raster\Jun12_1.tif
  Canopy area (m²): 2.218287488349865
  Soil DSM P10:      288.5849914550781
  Canopy DSM P90:    288.95001220703125
  Height (P90-P10):  0.365020751953125 m
Shapefiles F:\2025\BrunerYT&earlyCAD\process_folder\Jun12\output\plot_Jun12_10\canopy_Jun12_10.shp and F:\2025\BrunerYT&earlyCAD\process_folder\Jun12\output\plot_Jun12_10\soil_Jun12_10.shp created successfully.
Processing Zonal Statistics (P90 canopy - P10 soil):
  Soil Poly

  Canopy area (m²): 2.2765983549061235
  Soil DSM P10:      305.8659973144531
  Canopy DSM P90:    306.27398681640625
  Height (P90-P10):  0.407989501953125 m
Shapefiles D:\2025\BerkeyYT\process\June13\output\plot_June13_31\canopy_June13_31.shp and D:\2025\BerkeyYT\process\June13\output\plot_June13_31\soil_June13_31.shp created successfully.
Processing Zonal Statistics (P90 canopy - P10 soil):
  Soil Polygon:   D:\2025\BerkeyYT\process\June13\output\plot_June13_31\soil_June13_31.shp
  Canopy Polygon: D:\2025\BerkeyYT\process\June13\output\plot_June13_31\canopy_June13_31.shp
  DSM Image:      D:\2025\BerkeyYT\process\June13\output\split_dsm_raster\June13_31.TIF
  Canopy area (m²): 3.30259121993069
  Soil DSM P10:      305.8949890136719
  Canopy DSM P90:    306.35150146484375
  Height (P90-P10):  0.456512451171875 m
Shapefiles D:\2025\BerkeyYT\process\June13\output\plot_June13_32\canopy_June13_32.shp and D:\2025\BerkeyYT\process\June13\output\plot_June13_32\soil_June13_32.shp created suc

  Canopy area (m²): 3.897121954959263
  Soil DSM P10:      307.20233154296875
  Canopy DSM P90:    307.55499267578125
  Height (P90-P10):  0.3526611328125 m
Shapefiles D:\2025\BerkeyYT\process\June18\output\plot_June18_11\canopy_June18_11.shp and D:\2025\BerkeyYT\process\June18\output\plot_June18_11\soil_June18_11.shp created successfully.
Processing Zonal Statistics (P90 canopy - P10 soil):
  Soil Polygon:   D:\2025\BerkeyYT\process\June18\output\plot_June18_11\soil_June18_11.shp
  Canopy Polygon: D:\2025\BerkeyYT\process\June18\output\plot_June18_11\canopy_June18_11.shp
  DSM Image:      D:\2025\BerkeyYT\process\June18\output\split_dsm_raster\June18_11.TIF
  Canopy area (m²): 2.2579126098081
  Soil DSM P10:      307.2539978027344
  Canopy DSM P90:    307.4800109863281
  Height (P90-P10):  0.22601318359375 m
Shapefiles D:\2025\BerkeyYT\process\June18\output\plot_June18_12\canopy_June18_12.shp and D:\2025\BerkeyYT\process\June18\output\plot_June18_12\soil_June18_12.shp created successf

...

Shapefiles D:\2025\BerkeyYT\process\May14\output\plot_May14_23\canopy_May14_23.shp and D:\2025\BerkeyYT\process\May14\output\plot_May14_23\soil_May14_23.shp created successfully.
Processing Zonal Statistics (P90 canopy - P10 soil):
  Soil Polygon:   D:\2025\BerkeyYT\process\May14\output\plot_May14_23\soil_May14_23.shp
  Canopy Polygon: D:\2025\BerkeyYT\process\May14\output\plot_May14_23\canopy_May14_23.shp
  DSM Image:      D:\2025\BerkeyYT\process\May14\output\split_dsm_raster\May14_23.TIF
  Canopy area (m²): 0.23478883509446397
  Soil DSM P10:      307.3290100097656
  Canopy DSM P90:    307.3909912109375
  Height (P90-P10):  0.061981201171875 m
Shapefiles D:\2025\BerkeyYT\process\May14\output\plot_May14_24\canopy_May14_24.shp and D:\2025\BerkeyYT\process\May14\output\plot_May14_24\soil_May14_24.shp created successfully.
Processing Zonal Statistics (P90 canopy - P10 soil):
  Soil Polygon:   D:\2025\BerkeyYT\process\May14\output\plot_May14_24\soil_May14_24.shp
  Canopy Polygon: D:\2025

  Canopy area (m²): 0.39054645007380756
  Soil DSM P10:      306.9930114746094
  Canopy DSM P90:    307.0799865722656
  Height (P90-P10):  0.08697509765625 m
Shapefiles D:\2025\BerkeyYT\process\May14\output\plot_May14_5\canopy_May14_5.shp and D:\2025\BerkeyYT\process\May14\output\plot_May14_5\soil_May14_5.shp created successfully.
Processing Zonal Statistics (P90 canopy - P10 soil):
  Soil Polygon:   D:\2025\BerkeyYT\process\May14\output\plot_May14_5\soil_May14_5.shp
  Canopy Polygon: D:\2025\BerkeyYT\process\May14\output\plot_May14_5\canopy_May14_5.shp
  DSM Image:      D:\2025\BerkeyYT\process\May14\output\split_dsm_raster\May14_5.TIF
  Canopy area (m²): 0.6235035597483225
  Soil DSM P10:      307.031005859375
  Canopy DSM P90:    307.1499938964844
  Height (P90-P10):  0.118988037109375 m
Shapefiles D:\2025\BerkeyYT\process\May14\output\plot_May14_6\canopy_May14_6.shp and D:\2025\BerkeyYT\process\May14\output\plot_May14_6\soil_May14_6.shp created successfully.
Processing Zonal Statis

Shapefiles D:\2025\BerkeyYT\process\May29\output\plot_May29_15\canopy_May29_15.shp and D:\2025\BerkeyYT\process\May29\output\plot_May29_15\soil_May29_15.shp created successfully.
Processing Zonal Statistics (P90 canopy - P10 soil):
  Soil Polygon:   D:\2025\BerkeyYT\process\May29\output\plot_May29_15\soil_May29_15.shp
  Canopy Polygon: D:\2025\BerkeyYT\process\May29\output\plot_May29_15\canopy_May29_15.shp
  DSM Image:      D:\2025\BerkeyYT\process\May29\output\split_dsm_raster\May29_15.TIF
  Canopy area (m²): 1.2762709894942255
  Soil DSM P10:      307.15899658203125
  Canopy DSM P90:    307.3320007324219
  Height (P90-P10):  0.173004150390625 m
Shapefiles D:\2025\BerkeyYT\process\May29\output\plot_May29_16\canopy_May29_16.shp and D:\2025\BerkeyYT\process\May29\output\plot_May29_16\soil_May29_16.shp created successfully.
Processing Zonal Statistics (P90 canopy - P10 soil):
  Soil Polygon:   D:\2025\BerkeyYT\process\May29\output\plot_May29_16\soil_May29_16.shp
  Canopy Polygon: D:\2025

  Canopy area (m²): 1.1579641798142208
  Soil DSM P10:      307.3269958496094
  Canopy DSM P90:    307.5050048828125
  Height (P90-P10):  0.178009033203125 m
Shapefiles D:\2025\BerkeyYT\process\May29\output\plot_May29_27\canopy_May29_27.shp and D:\2025\BerkeyYT\process\May29\output\plot_May29_27\soil_May29_27.shp created successfully.
Processing Zonal Statistics (P90 canopy - P10 soil):
  Soil Polygon:   D:\2025\BerkeyYT\process\May29\output\plot_May29_27\soil_May29_27.shp
  Canopy Polygon: D:\2025\BerkeyYT\process\May29\output\plot_May29_27\canopy_May29_27.shp
  DSM Image:      D:\2025\BerkeyYT\process\May29\output\split_dsm_raster\May29_27.TIF
  Canopy area (m²): 0.44915123003004737
  Soil DSM P10:      307.2876281738281
  Canopy DSM P90:    307.4097900390625
  Height (P90-P10):  0.122161865234375 m
Shapefiles D:\2025\BerkeyYT\process\May29\output\plot_May29_28\canopy_May29_28.shp and D:\2025\BerkeyYT\process\May29\output\plot_May29_28\soil_May29_28.shp created successfully.
Processi

  Canopy area (m²): 0.9845561550881466
  Soil DSM P10:      307.2239990234375
  Canopy DSM P90:    307.4100036621094
  Height (P90-P10):  0.186004638671875 m
Shapefiles D:\2025\BerkeyYT\process\May29\output\plot_May29_9\canopy_May29_9.shp and D:\2025\BerkeyYT\process\May29\output\plot_May29_9\soil_May29_9.shp created successfully.
Processing Zonal Statistics (P90 canopy - P10 soil):
  Soil Polygon:   D:\2025\BerkeyYT\process\May29\output\plot_May29_9\soil_May29_9.shp
  Canopy Polygon: D:\2025\BerkeyYT\process\May29\output\plot_May29_9\canopy_May29_9.shp
  DSM Image:      D:\2025\BerkeyYT\process\May29\output\split_dsm_raster\May29_9.TIF
  Canopy area (m²): 1.4850854155285313
  Soil DSM P10:      307.1719970703125
  Canopy DSM P90:    307.3940124511719
  Height (P90-P10):  0.222015380859375 m
  -> Saved per-date CSV: D:\2025\BerkeyYT\process\May29\output\May29_plotwise_canopy_height_and_VI.csv
  -> Saved date-master copy: D:\2025\BerkeyYT\process\date_masters\May29_plotwise_canopy_heigh